In [1]:
print("Jay Ganesh")

Jay Ganesh


In [2]:
import os


In [ ]:
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch18 FirstMLOPs\\FirstMLOPsProject\\research'

In [ ]:
# os.chdir('../')

In [ ]:
# %pwd

'd:\\ProjectAarya\\MLOPs\\Code\\Ch18 FirstMLOPs\\FirstMLOPsProject'

In [8]:
import pandas as pd
data=pd.read_csv('artifacts/data_ingest/data.csv')
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [10]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [11]:
data.shape

(4898, 12)

In [15]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

    

In [16]:
from src.First_End_to_End_Project.constants import *
from src.First_End_to_End_Project.utils.common import read_yaml,create_directories

In [17]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):   
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath) 
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def datavalidationconfig(self)->DataValidationConfig:
        config=self.config.data_validation
        schema=self.schema.COLUMNS

        create_directories([config.root_dir])
        data_validation_config= DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema
        )

        return data_validation_config


In [18]:
import os
from src.First_End_to_End_Project import logger


In [27]:
class DataValidation:
    def __init__(self,config:DataValidationConfig):
        self.config=config

    def validate_all_columns(self)->bool:
        try:
            validation_status=None

            data=pd.read_csv(self.config.unzip_data_dir)
            all_cols=list(data.columns)
            all_schema=list(self.config.all_schema.keys())

            for col in all_cols:
                if col not in all_schema:
                    validation_status=False
                    with open(self.config.STATUS_FILE,'w') as f:
                        f.write(f"Validation Status: {validation_status}")
                elif str(data[col].dtype)!=self.config.all_schema[col]:
                    validation_status=False
                    with open(self.config.STATUS_FILE,'w') as f:
                        f.write(f"Validation Status: {validation_status} | Reason: Column '{col}' "
                            f"expected dtype '{self.config.all_schema[col]}' "
                            f"but got '{str(data[col].dtype)}'"
                        )
                else:
                    validation_status=True
                    with open(self.config.STATUS_FILE,'w') as f:
                        f.write(f"Validation Status: {validation_status}")
            return validation_status
        except Exception as e:
            raise e

In [28]:
try:
    config=ConfigurationManager()
    data_validation_config=config.datavalidationconfig()
    data_validation=DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-03-18 13:47:08,710:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\config\config.yaml loaded successfully]
[2026-03-18 13:47:08,713:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\params.yaml loaded successfully]
[2026-03-18 13:47:08,717:INFO:common:yaml file: D:\ProjectAarya\MLOPs\Code\Ch18 FirstMLOPs\FirstMLOPsProject\schema.yaml loaded successfully]
[2026-03-18 13:47:08,719:INFO:common:created directory at: artifacts]
[2026-03-18 13:47:08,721:INFO:common:created directory at: artifacts/data_validation]
